In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.formula.api as sm
import os
from scipy.stats import shapiro
from scipy.stats import levene
from scipy.stats import kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
folder = '/content/drive/MyDrive/data_science'
df = pd.read_csv('/content/drive/MyDrive/data_science/output/dataset_final.csv')


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14268 entries, 0 to 14267
Data columns (total 41 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   award_id                          14268 non-null  str    
 1   release_id                        14268 non-null  str    
 2   fecha_adj                         14268 non-null  str    
 3   monto_adj                         14268 non-null  int64  
 4   log_monto                         14268 non-null  float64
 5   anio                              14268 non-null  int64  
 6   mes                               14268 non-null  int64  
 7   contract_id                       14264 non-null  str    
 8   fecha_firma                       14264 non-null  str    
 9   plazo_dias                        14264 non-null  float64
 10  ruc                               14268 non-null  str    
 11  proveedor                         14268 non-null  str    
 12  prov_n_previas 

In [ ]:
def verificar_p_value(p_value, significancia=0.05):
  if p_value < significancia:
    print("Rechazar la hipótesis nula")
  else:
    print("No hay evidencia suficiente para rechazar la hipótesis nula")

In [ ]:
datos = df.copy()
datos["hubo_ampl_monto"] = (datos["ratio_ampliacion"] > 0).astype(int)
variables = [
    "ratio_ampliacion",
    "hubo_ampl_plazo",
    "categoria_principal",
    "metodo_contratacion",
    "log_monto",
    "plazo_dias",
    "n_enmiendas",
    "prov_n_previas",
    "prov_antiguedad_dias",
    "dias_pub_a_adj"
]
datos[variables].describe(include="all")

,ratio_ampliacion,hubo_ampl_plazo,categoria_principal,metodo_contratacion,log_monto,plazo_dias,n_enmiendas,prov_n_previas,prov_antiguedad_dias,dias_pub_a_adj
count,14268.000000,14268.000000,14268,14268,14268.000000,14264.000000,14268.000000,14268.000000,14268.000000,14268.000000
unique,NaN,NaN,3,2,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,goods,open,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,9083,14001,NaN,NaN,NaN,NaN,NaN,NaN
mean,0.018113,0.051093,NaN,NaN,17.632380,183.621845,0.260233,9.839080,126.882885,61.018994
std,0.118270,0.220196,NaN,NaN,2.026246,200.159680,0.702196,15.698106,112.131397,47.568833
min,0.000000,0.000000,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,NaN,NaN,16.239612,43.000000,0.000000,0.000000,0.000000,29.000000
50%,0.000000,0.000000,NaN,NaN,17.620853,95.000000,0.000000,3.000000,119.000000,48.000000
75%,0.000000,0.000000,NaN,NaN,18.890684,300.000000,0.000000,12.000000,224.000000,80.000000


In [ ]:
tabla = pd.crosstab(datos["metodo_contratacion"], datos["hubo_ampl_monto"] )

# **Ampliación de plazo vs. ampliación monetaria**

H₀: No existe diferencia estadísticamente significativa en el nivel de ampliación monetaria entre las obras con y sin ampliación de plazo.

μsin ampl. plazo = μcon ampl. plazo

H₁: Existe una diferencia estadísticamente significativa en el nivel de ampliación monetaria entre las obras con y sin ampliación de plazo.

μsin ampl. plazo != μcon ampl. plazo
	​

In [ ]:
d1 = datos[["ratio_ampliacion", "hubo_ampl_plazo"]].copy()
d1["ratio_ampliacion"] = pd.to_numeric(d1["ratio_ampliacion"], errors="coerce")
d1 = d1.dropna()
d1 = d1[np.isfinite(d1["ratio_ampliacion"])]

grupo_sin = d1.loc[d1["hubo_ampl_plazo"] == 0, "ratio_ampliacion"]
grupo_con = d1.loc[d1["hubo_ampl_plazo"] == 1, "ratio_ampliacion"]

In [ ]:
welch = stats.ttest_ind(grupo_con, grupo_sin, equal_var=False)

print("t de Welch")
print("t =", welch.statistic)
print("p-value =", welch.pvalue)
verificar_p_value(p_value=welch.pvalue)


--- Prueba t de Welch ---
t = 6.118763107717013
p-value = 1.4464947809343175e-09
Rechazar la hipótesis nula


In [ ]:
mw = stats.mannwhitneyu(
    grupo_con, grupo_sin, alternative="two-sided"
)

print("Mann-Whitney")
print("U =", mw.statistic)
print("p-value =", mw.pvalue)
verificar_p_value(p_value=mw.pvalue)


--- Mann-Whitney ---
U = 5749504.0
p-value = 1.8409731786441851e-44
Rechazar la hipótesis nula


Se realizaron una prueba t de Welch y una prueba no paramétrica de Mann-Whitney, utilizando un nivel de significancia de 0,05. De los contratos analizados, 13.539 no presentaron ampliación de plazo, mientras que 729 sí la presentaron. Ambas pruebas mostraron diferencias estadísticamente significativas entre los grupos (t=-5,067; p < 0,001; U = 1.531.495,5; p < 0,001), por lo que se rechazó la hipótesis nula. Estos resultados indican que existe evidencia de diferencias en el ratio de ampliación monetaria según la presencia de ampliaciones de plazo. No obstante, para determinar qué grupo presenta mayores valores y evaluar la magnitud práctica de la diferencia, es necesario comparar sus medias y medianas. Asimismo, los resultados reflejan una asociación estadística, pero no demuestran que la ampliación de plazo cause una ampliación monetaria.

# **Diferencias entre categorías de obra**
H₀: El nivel medio de ampliación monetaria es igual entre las diferentes categorías principales de contratación.

H₁: Al menos una categoría presenta un nivel medio de ampliación monetaria diferente.

In [ ]:
!pip -q install pingouin scikit-posthocs
import pingouin as pg
import scikit_posthocs as sp

In [ ]:
d2 = datos[["ratio_ampliacion", "categoria_principal"]].dropna()

In [ ]:
grupos_anova = [grupo["ratio_ampliacion"].to_numpy() for _, grupo in d2.groupby("categoria_principal")]
levene_anova = levene(*grupos_anova, center="median")

print("Levene:")
print("Estadístico =", levene_anova.statistic)
print("p-value =", levene_anova.pvalue)

Levene:
Estadístico = 52.105344197213626
p-value = 2.8391784244996174e-23


In [ ]:
alpha = 0.05
welch = pg.welch_anova(data=d2, dv="ratio_ampliacion", between="categoria_principal")
print("Anova Welch:")
print(welch.to_string(index=False))

p_welch = welch['p_unc'][0]

Anova Welch:
             Source  ddof1       ddof2         F        p_unc      np2
categoria_principal      2 2719.108345 93.128191 7.585575e-40 0.007252


In [ ]:
if p_welch < alpha:
    games_howell = pg.pairwise_gameshowell(data=d2, dv="ratio_ampliacion", between="categoria_principal")

    print("hoc Games-Howell")
    print(games_howell.sort_values("pval").round(6).to_string(index=False))

    print("Presenta diferencias estadísticamente significativas.")
else:
    print("El Anova de Welch no fue significativo")

hoc Games-Howell
       A        B   mean_A   mean_B      diff       se          T          df     pval    hedges
   goods    works 0.010665 0.037234 -0.026569 0.002100 -12.649305 1363.761794 0.000000 -0.571967
   goods services 0.010665 0.029232 -0.018567 0.003398  -5.463477 4066.427161 0.000000 -0.153050
services    works 0.029232 0.037234 -0.008002 0.003947  -2.027302 5157.663842 0.105788 -0.042647
Presenta diferencias estadísticamente significativas.


In [ ]:
kw = stats.kruskal(*grupos_anova)

print("Kruskal-Wallis")
print(f"H = {kw.statistic:.6f}")
print(f"p-value = {kw.pvalue:.6g}")

n = len(d2)
k = d2["categoria_principal"].nunique()

epsilon2 = (kw.statistic - k + 1) / (n - k)
epsilon2 = max(0, epsilon2)

print(f"Epsilon cuadrado = {epsilon2:.6f}")


--- Kruskal-Wallis ---
H = 395.249885
p-value = 1.48791e-86
Epsilon cuadrado = 0.027567


In [ ]:
if kw.pvalue < alpha:
    dunn = sp.posthoc_dunn(d2, val_col="ratio_ampliacion", group_col="categoria_principal", p_adjust="holm")

    print("hoc de Dunn")
    print(dunn.round(6).to_string())

    print("Presenta diferencias estadísticamente significativas.")
else:
    print("Kruskal-Wallis no fue significativo.")


--- Post hoc de Dunn (p ajustados por Holm) ---
          goods  services  works
goods       1.0       0.0    0.0
services    0.0       1.0    0.0
works       0.0       0.0    1.0

En la matriz de Dunn, cada celda es el p-value ajustado para comparar dos categorías. Valores < 0.05 indican diferencias estadísticamente significativas.


Se aplicó la prueba de Levene, que detectó diferencias significativas entre las varianzas de las categorías W=52,105; p<0,001, por lo que se utilizó el ANOVA de Welch. Este identificó diferencias estadísticamente significativas en el ratio de ampliación según la categoría principal F(2;2719,11)=93,128, p<0,001, aunque el tamaño del efecto global fue pequeño np2=0,0073. En las comparaciones post hoc de Games-Howell, las categorías goods y services presentaron medias inferiores a works, con diferencias significativas respecto de goods, pero no entre services y works. Kruskal-Wallis también detectó diferencias en las distribuciones (H=395, 250), p<0,001, con un tamaño de efecto pequeño epsilon cuadrado=0,0276.

# **Método de contratación y presencia de ampliación**
H₀: La presencia de ampliaciones monetarias es independiente del método de contratación utilizado.

H₁: Existe una asociación estadísticamente significativa entre el método de contratación y la presencia de ampliaciones monetarias.

In [ ]:
chi2, p, gl, esperadas = stats.chi2_contingency(tabla)

print("Chi-cuadrado =", chi2)
print("gl =", gl)
print("p-value =", p)

Chi-cuadrado = 1.5236533381381545
gl = 1
p-value = 0.21706747034364757


In [ ]:
esperadas_df = pd.DataFrame(
    esperadas,
    index=tabla.index,
    columns=tabla.columns
)

print(esperadas_df)

hubo_ampl_monto                 0            1
metodo_contratacion                           
direct                 238.312658    28.687342
open                 12496.687342  1504.312658


In [ ]:
print(
    "Mínimo esperado:",
    esperadas.min()
)

print(
    "Celdas con esperado < 5:",
    (esperadas < 5).sum()
)

Mínimo esperado: 28.687342304457527
Celdas con esperado < 5: 0


In [ ]:
n = tabla.to_numpy().sum()
phi2 = chi2 / n
r, c = tabla.shape
cramers_v = np.sqrt(phi2 / min(c - 1, r - 1))

print("V de Cramér =", cramers_v)

V de Cramér = 0.010333835250132555


Se aplicó una prueba chi-cuadrado de independencia para evaluar la relación entre el método de contratación y la presencia de ampliación monetaria, utilizando un nivel de significancia de 0,05. Se obtuvo un estadístico de 1,524, con un valor p de 0,217, por lo que no se rechazó la hipótesis nula de independencia. Los supuestos relativos a las frecuencias esperadas fueron adecuados, dado que ninguna celda presentó una frecuencia esperada inferior a 5. Asimismo, V de Cramér fue de 0,0103, lo que indica una asociación observada prácticamente nula. En consecuencia, los resultados no aportan evidencia estadística suficiente de una asociación entre ambas variables en el conjunto analizado. Esta conclusión no demuestra que el método de contratación carezca de relación con las ampliaciones en todos los contextos, ni permite establecer relaciones causales.